In [1]:
# For text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# For topic modeling
from gensim import corpora
from gensim.models import LdaModel

# For dataframe display
import pandas as pd

# Download NLTK resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/59281338-7aa6-4b99-8f53-e59317ae67a7/nltk_data..
[nltk_data]     .
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/59281338-7aa6-4b99-8f53-e59317ae67a7/nltk_data..
[nltk_data]     .
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/59281338-7aa6-4b99-8f53-e59317ae67a7/nltk_data..
[nltk_data]     .
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
documents = [
    "Rafael Nadal Joins Roger Federer in Missing U.S. Open",
    "Rafael Nadal Is Out of the Australian Open",
    "Biden Announces Virus Measures",
    "Biden's Virus Plans Meet Reality",
    "Where Biden's Virus Plan Stands"
]

documents

['Rafael Nadal Joins Roger Federer in Missing U.S. Open',
 'Rafael Nadal Is Out of the Australian Open',
 'Biden Announces Virus Measures',
 "Biden's Virus Plans Meet Reality",
 "Where Biden's Virus Plan Stands"]

In [3]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())                 # convert to lowercase and tokenize
    tokens = [token for token in tokens if token.isalnum()]   # remove non-alphanumeric
    tokens = [token for token in tokens if token not in stop_words]  # remove stopwords
    tokens = [lemmatizer.lemmatize(token) for token in tokens]       # lemmatize
    return tokens

preprocessed_documents = [preprocess_text(doc) for doc in documents]
preprocessed_documents

[['rafael', 'nadal', 'join', 'roger', 'federer', 'missing', 'open'],
 ['rafael', 'nadal', 'australian', 'open'],
 ['biden', 'announces', 'virus', 'measure'],
 ['biden', 'virus', 'plan', 'meet', 'reality'],
 ['biden', 'virus', 'plan', 'stand']]

In [4]:
dictionary = corpora.Dictionary(preprocessed_documents)
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

print("Dictionary:")
print(dictionary.token2id)

print("\nCorpus:")
print(corpus)

Dictionary:
{'federer': 0, 'join': 1, 'missing': 2, 'nadal': 3, 'open': 4, 'rafael': 5, 'roger': 6, 'australian': 7, 'announces': 8, 'biden': 9, 'measure': 10, 'virus': 11, 'meet': 12, 'plan': 13, 'reality': 14, 'stand': 15}

Corpus:
[[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1)], [(3, 1), (4, 1), (5, 1), (7, 1)], [(8, 1), (9, 1), (10, 1), (11, 1)], [(9, 1), (11, 1), (12, 1), (13, 1), (14, 1)], [(9, 1), (11, 1), (13, 1), (15, 1)]]


In [5]:
lda_model = LdaModel(
    corpus=corpus,
    num_topics=2,
    id2word=dictionary,
    passes=15
)

In [6]:
article_labels = []

for i, doc in enumerate(preprocessed_documents):
    bow = dictionary.doc2bow(doc)
    topics = lda_model.get_document_topics(bow)
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    article_labels.append(dominant_topic)

df = pd.DataFrame({
    "Article": documents,
    "Topic": article_labels
})

print("Table with Articles and Topic:")
print(df)

Table with Articles and Topic:
                                             Article  Topic
0  Rafael Nadal Joins Roger Federer in Missing U....      0
1         Rafael Nadal Is Out of the Australian Open      0
2                     Biden Announces Virus Measures      1
3                   Biden's Virus Plans Meet Reality      0
4                    Where Biden's Virus Plan Stands      0


In [7]:
print("Top Terms for Each Topic:")

for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}:")
    terms = [term.strip() for term in topic.split("+")]
    for term in terms:
        weight, word = term.split("*")
        print(f"- {word.strip()} (weight: {weight.strip()})")
    print()

Top Terms for Each Topic:
Topic 0:
- "rafael" (weight: 0.092)
- "nadal" (weight: 0.091)
- "open" (weight: 0.091)
- "plan" (weight: 0.090)
- "virus" (weight: 0.082)
- "biden" (weight: 0.081)
- "federer" (weight: 0.055)
- "roger" (weight: 0.055)
- "join" (weight: 0.055)
- "missing" (weight: 0.055)

Topic 1:
- "biden" (weight: 0.139)
- "virus" (weight: 0.139)
- "announces" (weight: 0.117)
- "measure" (weight: 0.117)
- "stand" (weight: 0.044)
- "plan" (weight: 0.043)
- "reality" (weight: 0.041)
- "meet" (weight: 0.041)
- "australian" (weight: 0.040)
- "open" (weight: 0.040)



In [8]:
print("Topic 0 seems to be related around politics and virus.")
print("Topic 1 seems to be related to tennis.")

Topic 0 seems to be related around politics and virus.
Topic 1 seems to be related to tennis.
